# Anthropic extraction workflow

This template builds a small corpus and extracts structured records with Anthropic text and vision profiles. Set `ANTHROPIC_API_KEY` before starting Jupyter and choose a Messages API model available to your account.

In [ ]:
%env PMT_DB=papers.db
%env PMT_QUERY=lithium solid electrolyte
%env PMT_RECIPE=sse
%env PMT_MODEL=YOUR_ANTHROPIC_MODEL
%env PMT_OUTPUT=temp_anthropic_materials.csv
%env PMT_FINAL=anthropic_materials.csv

## Configure model profiles

Use separate identifiers if the chosen text model does not accept images. `pmt config status` shows the effective provider, endpoint, and capabilities without printing the secret.

In [ ]:
%%bash
set -euo pipefail
test "$PMT_MODEL" != "YOUR_ANTHROPIC_MODEL"
pmt config model text --provider anthropic --model "$PMT_MODEL"
pmt config model vision --provider anthropic --model "$PMT_MODEL"
pmt config status

## Build the corpus

Search and download credentials are independent of the Anthropic key. This example uses OpenAlex discovery and every configured download source.

In [ ]:
%%bash
set -euo pipefail
pmt search "$PMT_QUERY" "$PMT_DB" --source openalex --count 25
pmt download "$PMT_DB" --format both
pmt corpus stats "$PMT_DB"

## Extract and store records

Begin with five papers. Review the intermediate CSV before increasing the count or using `--force` for a deliberate rerun.

In [ ]:
%%bash
set -euo pipefail
pmt scrape "$PMT_DB" "$PMT_RECIPE" --mode text-images --image-context paper-text --count 5 --output "$PMT_OUTPUT"
pmt store "$PMT_DB" "$PMT_OUTPUT" "$PMT_FINAL" "$PMT_RECIPE" --assume-yes
pmt status "$PMT_DB"